# Project: Wrangling and Analyze Data

In [1]:
#import libraries
import pandas as pd
import requests
import numpy as np

## Data Gathering
In the cell below, gather **all** three pieces of data for this project and load them in the notebook. **Note:** the methods required to gather each data are different.
1. Directly download the WeRateDogs Twitter archive data (twitter_archive_enhanced.csv)

In [2]:
#import csv file and read the top 5 rows
twit_arch = pd.read_csv('twitter-archive-enhanced.csv')
twit_arch.head()

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,rating_numerator,rating_denominator,name,doggo,floofer,pupper,puppo
0,892420643555336193,NaN,NaN,2017-08-01 16:23:56 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Phineas. He's a mystical boy. Only eve...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892420643...,13,10,Phineas,None,None,None,None
1,892177421306343426,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,13,10,Tilly,None,None,None,None
2,891815181378084864,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,12,10,Archie,None,None,None,None
3,891689557279858688,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,13,10,Darla,None,None,None,None
4,891327558926688256,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,12,10,Franklin,None,None,None,None


2. Use the Requests library to download the tweet image prediction (image_predictions.tsv)

In [3]:
#URL for tweet image prediction
url = 'https://d17h27t6h515a5.cloudfront.net/topher/2017/August/599fd2ad_image-predictions/image-predictions.tsv'
response = requests.get(url)

with open('image-predictions.tsv', mode='wb') as file:
    file.write(response.content)
    #write data from file at URL to df "image_predictions"
    
image_predictions = pd.read_csv('image-predictions.tsv', sep = "\t")
image_predictions.head()

,tweet_id,jpg_url,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
0,666020888022790149,https://pbs.twimg.com/media/CT4udn0WwAA0aMy.jpg,1,Welsh_springer_spaniel,0.465074,True,collie,0.156665,True,Shetland_sheepdog,0.061428,True
1,666029285002620928,https://pbs.twimg.com/media/CT42GRgUYAA5iDo.jpg,1,redbone,0.506826,True,miniature_pinscher,0.074192,True,Rhodesian_ridgeback,0.072010,True
2,666033412701032449,https://pbs.twimg.com/media/CT4521TWwAEvMyu.jpg,1,German_shepherd,0.596461,True,malinois,0.138584,True,bloodhound,0.116197,True
3,666044226329800704,https://pbs.twimg.com/media/CT5Dr8HUEAA-lEu.jpg,1,Rhodesian_ridgeback,0.408143,True,redbone,0.360687,True,miniature_pinscher,0.222752,True
4,666049248165822465,https://pbs.twimg.com/media/CT5IQmsXIAAKY4A.jpg,1,miniature_pinscher,0.560311,True,Rottweiler,0.243682,True,Doberman,0.154629,True


3. Use the Tweepy library to query additional data via the Twitter API (tweet_json.txt)

In [4]:
#import tweepy library
import tweepy as tpy

#store keys and secrets for access to API
consumer_key = 'WVKfGEblnq9CR9JKFUvbDkujI' # 1BIkx8nnbEYnvFm5miTVvgpUy
consumer_secret = 'Ow9VCBZCb9HCoudnaIIhOApyThr77zwN5Y4UPoksXGq6aR88bu' # AGuqQSfJtXvZlU3nv35EkaUX9cRcJ3llVhaarSCM6VghtzMqgc
access_token = '1637157618144665600-5ssNm6NplvL5DxRUkYdoh9y1ixrZzO'
access_secret = 'UyHgVY2DFbt2ItYgQPj4VHlUyPl3D3egAD37GbrVrVTjz'

In [5]:
#authorize access to twitter data via API
def twit_auth():
    auth = tpy.OAuthHandler(consumer_key, consumer_secret)
    auth.set_access_token(access_token,access_secret)
    
    api = tpy.API(auth,wait_on_rate_limit=True,wait_on_rate_limit_notify=True)
    return api

api = twit_auth()

### Quality issues
Referring to the assessment of twit_arch and image_predictions 

1. Data contains retweets, which need to be removed

2. Data contains non-dogs, which need to be removed

3. There are entries where the dog's name is incomplete instead of NA

4. After merging the data, values in the p1_dog, p2_dog, and p3_dog columns were converted to float - they need to be bool.

5. Data should only contain observations with associated pictures

6. Inconsistent formatting - All columns containing status_id should be a string

7. Inconsistent formatting - All columns containing tweet_id should be a string

8. Inconsistent formatting - All columns containing user_id should be string

9. *BONUS* (Accidental additional resolve) - Inconsistent formatting - All columns containing dogs breeds or otherwise (p1, p2, p3) do not have the same case style. Each value found in these columns should either have a capitalized first letter or be all lowercase.


### Tidiness issues
1. "A single observational unit is stored in multiple tables" - The data required for this study is in two separate files. The data from these files can be joined together via column tweet_id

2. There are extraneous columns that can be deleted - doggo, floofer, pupper, puppo

In [6]:
twit_arch

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,rating_numerator,rating_denominator,name,doggo,floofer,pupper,puppo
0,892420643555336193,NaN,NaN,2017-08-01 16:23:56 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Phineas. He's a mystical boy. Only eve...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892420643...,13,10,Phineas,None,None,None,None
1,892177421306343426,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,13,10,Tilly,None,None,None,None
2,891815181378084864,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,12,10,Archie,None,None,None,None
3,891689557279858688,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,13,10,Darla,None,None,None,None
4,891327558926688256,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,12,10,Franklin,None,None,None,None
5,891087950875897856,NaN,NaN,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891087950...,13,10,None,None,None,None,None
6,890971913173991426,NaN,NaN,2017-07-28 16:27:12 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Meet Jax. He enjoys ice cream so much he gets ...,NaN,NaN,NaN,"https://gofundme.com/ydvmve-surgery-for-jax,ht...",13,10,Jax,None,None,None,None
7,890729181411237888,NaN,NaN,2017-07-28 00:22:40 +0000,"<a href=""http://twitter.com/download/iphone"" r...",When you watch your owner call another dog a g...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890729181...,13,10,None,None,None,None,None
8,890609185150312448,NaN,NaN,2017-07-27 16:25:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Zoey. She doesn't want to be one of th...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890609185...,13,10,Zoey,None,None,None,None
9,890240255349198849,NaN,NaN,2017-07-26 15:59:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Cassie. She is a college pup. Studying...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890240255...,14,10,Cassie,doggo,None,None,None


In [7]:
twit_arch.columns

Index(['tweet_id', 'in_reply_to_status_id', 'in_reply_to_user_id', 'timestamp',
       'source', 'text', 'retweeted_status_id', 'retweeted_status_user_id',
       'retweeted_status_timestamp', 'expanded_urls', 'rating_numerator',
       'rating_denominator', 'name', 'doggo', 'floofer', 'pupper', 'puppo'],
      dtype='object')

In [8]:
image_predictions

,tweet_id,jpg_url,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
0,666020888022790149,https://pbs.twimg.com/media/CT4udn0WwAA0aMy.jpg,1,Welsh_springer_spaniel,0.465074,True,collie,0.156665,True,Shetland_sheepdog,0.061428,True
1,666029285002620928,https://pbs.twimg.com/media/CT42GRgUYAA5iDo.jpg,1,redbone,0.506826,True,miniature_pinscher,0.074192,True,Rhodesian_ridgeback,0.072010,True
2,666033412701032449,https://pbs.twimg.com/media/CT4521TWwAEvMyu.jpg,1,German_shepherd,0.596461,True,malinois,0.138584,True,bloodhound,0.116197,True
3,666044226329800704,https://pbs.twimg.com/media/CT5Dr8HUEAA-lEu.jpg,1,Rhodesian_ridgeback,0.408143,True,redbone,0.360687,True,miniature_pinscher,0.222752,True
4,666049248165822465,https://pbs.twimg.com/media/CT5IQmsXIAAKY4A.jpg,1,miniature_pinscher,0.560311,True,Rottweiler,0.243682,True,Doberman,0.154629,True
5,666050758794694657,https://pbs.twimg.com/media/CT5Jof1WUAEuVxN.jpg,1,Bernese_mountain_dog,0.651137,True,English_springer,0.263788,True,Greater_Swiss_Mountain_dog,0.016199,True
6,666051853826850816,https://pbs.twimg.com/media/CT5KoJ1WoAAJash.jpg,1,box_turtle,0.933012,False,mud_turtle,0.045885,False,terrapin,0.017885,False
7,666055525042405380,https://pbs.twimg.com/media/CT5N9tpXIAAifs1.jpg,1,chow,0.692517,True,Tibetan_mastiff,0.058279,True,fur_coat,0.054449,False
8,666057090499244032,https://pbs.twimg.com/media/CT5PY90WoAAQGLo.jpg,1,shopping_cart,0.962465,False,shopping_basket,0.014594,False,golden_retriever,0.007959,True
9,666058600524156928,https://pbs.twimg.com/media/CT5Qw94XAAA_2dP.jpg,1,miniature_poodle,0.201493,True,komondor,0.192305,True,soft-coated_wheaten_terrier,0.082086,True


## Cleaning Data
In this section, clean **all** of the issues you documented while assessing. 

**Note:** Make a copy of the original data before cleaning. Cleaning includes merging individual pieces of data according to the rules of [tidy data](https://cran.r-project.org/web/packages/tidyr/vignettes/tidy-data.html). The result should be a high-quality and tidy master pandas DataFrame (or DataFrames, if appropriate).

In [9]:
# Make copies of original pieces of data
twit_arch_copy = twit_arch 
image_predictions_copy = image_predictions

### T - Issue #1
> #### "A single observational unit is stored in multiple tables" - The data required for this study is in two separate files. The data from these files can be joined together via column tweet_id

### QA - Issue #5
> #### Data should only contain observations with associated pictures

#### Code

In [10]:
#T - Issue #1 -The data required for this study is in two separate files- 
    #merge data together on column tweet_id
#QA - Issue #5 -Data should only contain observations with associated pictures - 
    #using inner join returns only tweets with images
twitter_data = twit_arch_copy.merge(image_predictions_copy, on='tweet_id')

#### Test

In [11]:
num_obs = str(twit_arch.shape[0])
print("num observations twit_arch: ", num_obs)

num_obs = str(image_predictions_copy.shape[0])
print("num observations image_predictions: ", num_obs)

num_obs = str(twitter_data.shape[0])
print("num observations merged tables (twitter_data): ", num_obs)

num observations twit_arch:  2356
num observations image_predictions:  2075
num observations merged tables (twitter_data):  2075


In [12]:
twitter_data.head()

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
0,892420643555336193,NaN,NaN,2017-08-01 16:23:56 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Phineas. He's a mystical boy. Only eve...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892420643...,...,1,orange,0.097049,False,bagel,0.085851,False,banana,0.076110,False
1,892177421306343426,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,...,1,Chihuahua,0.323581,True,Pekinese,0.090647,True,papillon,0.068957,True
2,891815181378084864,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,...,1,Chihuahua,0.716012,True,malamute,0.078253,True,kelpie,0.031379,True
3,891689557279858688,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,...,1,paper_towel,0.170278,False,Labrador_retriever,0.168086,True,spatula,0.040836,False
4,891327558926688256,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,...,2,basset,0.555712,True,English_springer,0.225770,True,German_short-haired_pointer,0.175219,True


In [13]:
twitter_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2075 entries, 0 to 2074
Data columns (total 28 columns):
tweet_id                      2075 non-null int64
in_reply_to_status_id         23 non-null float64
in_reply_to_user_id           23 non-null float64
timestamp                     2075 non-null object
source                        2075 non-null object
text                          2075 non-null object
retweeted_status_id           81 non-null float64
retweeted_status_user_id      81 non-null float64
retweeted_status_timestamp    81 non-null object
expanded_urls                 2075 non-null object
rating_numerator              2075 non-null int64
rating_denominator            2075 non-null int64
name                          2075 non-null object
doggo                         2075 non-null object
floofer                       2075 non-null object
pupper                        2075 non-null object
puppo                         2075 non-null object
jpg_url                       2075 

### QA - Issue #1
> #### Data contains retweets, which need to be removed

### QA - Issue #2
> #### Data contains non-dogs, which need to be removed

#### Code

In [14]:
#QA - Issue #1 - Data contains retweets, which need to be removed
    #number of retweets
num_retweets = twitter_data.where(twitter_data['retweeted_status_id'].notnull()).shape[0]
    #remove retweets
twitter_working_data = twitter_data.where(twitter_data['retweeted_status_id'].isnull())
    #num_retweets = twitter_working_data[twitter_working_data['retweeted_status_id'].notnull()] #count of retweets
print(num_retweets, " retweets in original dataset")

#QA - Issue #2 - Data contains non-dogs, which need to be removed
    #update twitter_working_data df to only include rows which have "True" if the image analysis includes a dog
twitter_working_data = twitter_working_data[(twitter_working_data['p1_dog']  == True) | (twitter_working_data['p2_dog']  == True) | (twitter_working_data['p3_dog']  == True)]
    #count number of non-dogs in original dataset
num_non_dogs = twitter_data[(twitter_data['p1_dog']  == False) & (twitter_data['p2_dog']  == False) & (twitter_data['p3_dog']  == False)].shape[0]
print(num_non_dogs, " non-dogs in original dataset")

2075  retweets in original dataset
324  non-dogs in original dataset


In [15]:
twitter_working_data

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
1,8.921774e+17,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,...,1.0,Chihuahua,0.323581,1.0,Pekinese,0.090647,1.0,papillon,0.068957,1.0
2,8.918152e+17,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,...,1.0,Chihuahua,0.716012,1.0,malamute,0.078253,1.0,kelpie,0.031379,1.0
3,8.916896e+17,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,...,1.0,paper_towel,0.170278,0.0,Labrador_retriever,0.168086,1.0,spatula,0.040836,0.0
4,8.913276e+17,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,...,2.0,basset,0.555712,1.0,English_springer,0.225770,1.0,German_short-haired_pointer,0.175219,1.0
5,8.910880e+17,NaN,NaN,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891087950...,...,1.0,Chesapeake_Bay_retriever,0.425595,1.0,Irish_terrier,0.116317,1.0,Indian_elephant,0.076902,0.0
6,8.909719e+17,NaN,NaN,2017-07-28 16:27:12 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Meet Jax. He enjoys ice cream so much he gets ...,NaN,NaN,NaN,"https://gofundme.com/ydvmve-surgery-for-jax,ht...",...,1.0,Appenzeller,0.341703,1.0,Border_collie,0.199287,1.0,ice_lolly,0.193548,0.0
7,8.907292e+17,NaN,NaN,2017-07-28 00:22:40 +0000,"<a href=""http://twitter.com/download/iphone"" r...",When you watch your owner call another dog a g...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890729181...,...,2.0,Pomeranian,0.566142,1.0,Eskimo_dog,0.178406,1.0,Pembroke,0.076507,1.0
8,8.906092e+17,NaN,NaN,2017-07-27 16:25:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Zoey. She doesn't want to be one of th...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890609185...,...,1.0,Irish_terrier,0.487574,1.0,Irish_setter,0.193054,1.0,Chesapeake_Bay_retriever,0.118184,1.0
9,8.902403e+17,NaN,NaN,2017-07-26 15:59:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Cassie. She is a college pup. Studying...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890240255...,...,1.0,Pembroke,0.511319,1.0,Cardigan,0.451038,1.0,Chihuahua,0.029248,1.0
10,8.900066e+17,NaN,NaN,2017-07-26 00:31:25 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Koda. He is a South Australian decksha...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890006608...,...,1.0,Samoyed,0.957979,1.0,Pomeranian,0.013884,1.0,chow,0.008167,1.0


#### Test

In [16]:
#count number of retweets in twitter_working_data df
num_retweets = twitter_working_data[twitter_working_data['retweeted_status_id'].notnull()].shape[0] #count of retweets
print(num_retweets, " retweets found.")

#count number of non-dogs in twitter_working_data df
num_non_dogs = twitter_working_data[(twitter_working_data['p1_dog']  == False) & (twitter_working_data['p2_dog']  == False) & (twitter_working_data['p3_dog']  == False)].shape[0]
print(num_non_dogs, " non-dogs found.")

0  retweets found.
0  non-dogs found.


### QA - Issue #3
> #### There are entries where the dog's name is incomplete instead of NA

#### Code

In [17]:
twitter_data['name']

0        Phineas
1          Tilly
2         Archie
3          Darla
4       Franklin
5           None
6            Jax
7           None
8           Zoey
9         Cassie
10          Koda
11         Bruno
12          None
13           Ted
14        Stuart
15        Oliver
16           Jim
17          Zeke
18       Ralphus
19        Canela
20        Gerald
21       Jeffrey
22          such
23        Canela
24          None
25          None
26          Maya
27        Mingus
28         Derek
29        Roscoe
          ...   
2045       quite
2046           a
2047        None
2048        None
2049        None
2050        None
2051        None
2052          an
2053           a
2054          an
2055        None
2056        None
2057        None
2058        None
2059        None
2060        None
2061        None
2062        None
2063        None
2064         the
2065         the
2066           a
2067           a
2068          an
2069           a
2070        None
2071           a
2072          

In [18]:
#Num dogs w/o names
lowercase_letters = twitter_working_data.loc[twitter_working_data['name'].str.match('^[a-z].*'),'name'].count()
lowercase_letters
print("lowercase_letters: ", lowercase_letters)

named_none = twitter_working_data[twitter_working_data['name'] == 'None']['name'].count()
print("None: ", named_none)

lowercase_letters:  80
None:  419


In [19]:
#QA Issue #3

#replace values that start with a lowercase letter with 'None'
#Error Avoidance: This method allows all non-names to be replaced with NaN in the next step, 
    #since a list with NA values cannot be indexed
twitter_working_data.loc[twitter_working_data['name'].str.match('^[a-z].*'),'name'] = 'None'

#replace all 'None' values with NaN
twitter_working_data.loc[twitter_working_data['name'].str.match('None'),'name'] = np.nan

#### Test

In [20]:
#count number of dogs with 'None' as name, should be 0
named_none = twitter_working_data[twitter_working_data['name'] == 'None']['name'].count()
print("New count of None: ", named_none)

#count number of dogs with NA as name
dogs_with_no_name = twitter_working_data['name'].isna().sum()
print("New count of NA: ", dogs_with_no_name)

twitter_working_data['name']

New count of None:  0
New count of NA:  499


1          Tilly
2         Archie
3          Darla
4       Franklin
5            NaN
6            Jax
7            NaN
8           Zoey
9         Cassie
10          Koda
11         Bruno
12           NaN
13           Ted
14        Stuart
15        Oliver
16           Jim
17          Zeke
18       Ralphus
20        Gerald
21       Jeffrey
23        Canela
24           NaN
25           NaN
26          Maya
27        Mingus
29        Roscoe
30       Waffles
31         Jimbo
32        Maisey
34           NaN
          ...   
2039         NaN
2040         NaN
2041         NaN
2042         NaN
2043         NaN
2044      Walter
2046         NaN
2047         NaN
2048         NaN
2050         NaN
2051         NaN
2052         NaN
2054         NaN
2055         NaN
2058         NaN
2059         NaN
2060         NaN
2061         NaN
2062         NaN
2063         NaN
2064         NaN
2065         NaN
2066         NaN
2067         NaN
2069         NaN
2070         NaN
2071         NaN
2072         N

### QA Issue #4
> #### After merging the data, values in the p1_dog, p2_dog, and p3_dog columns were converted to float instead of bool.

In [21]:
#Convert p1_dog, p2_dog, and p3_dog columns back to bool
twitter_working_data['p1_dog'] = twitter_working_data['p1_dog'].astype(bool)
twitter_working_data['p2_dog'] = twitter_working_data['p2_dog'].astype(bool)
twitter_working_data['p3_dog'] = twitter_working_data['p3_dog'].astype(bool)
twitter_working_data

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
1,8.921774e+17,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,...,1.0,Chihuahua,0.323581,True,Pekinese,0.090647,True,papillon,0.068957,True
2,8.918152e+17,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,...,1.0,Chihuahua,0.716012,True,malamute,0.078253,True,kelpie,0.031379,True
3,8.916896e+17,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,...,1.0,paper_towel,0.170278,False,Labrador_retriever,0.168086,True,spatula,0.040836,False
4,8.913276e+17,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,...,2.0,basset,0.555712,True,English_springer,0.225770,True,German_short-haired_pointer,0.175219,True
5,8.910880e+17,NaN,NaN,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891087950...,...,1.0,Chesapeake_Bay_retriever,0.425595,True,Irish_terrier,0.116317,True,Indian_elephant,0.076902,False
6,8.909719e+17,NaN,NaN,2017-07-28 16:27:12 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Meet Jax. He enjoys ice cream so much he gets ...,NaN,NaN,NaN,"https://gofundme.com/ydvmve-surgery-for-jax,ht...",...,1.0,Appenzeller,0.341703,True,Border_collie,0.199287,True,ice_lolly,0.193548,False
7,8.907292e+17,NaN,NaN,2017-07-28 00:22:40 +0000,"<a href=""http://twitter.com/download/iphone"" r...",When you watch your owner call another dog a g...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890729181...,...,2.0,Pomeranian,0.566142,True,Eskimo_dog,0.178406,True,Pembroke,0.076507,True
8,8.906092e+17,NaN,NaN,2017-07-27 16:25:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Zoey. She doesn't want to be one of th...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890609185...,...,1.0,Irish_terrier,0.487574,True,Irish_setter,0.193054,True,Chesapeake_Bay_retriever,0.118184,True
9,8.902403e+17,NaN,NaN,2017-07-26 15:59:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Cassie. She is a college pup. Studying...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890240255...,...,1.0,Pembroke,0.511319,True,Cardigan,0.451038,True,Chihuahua,0.029248,True
10,8.900066e+17,NaN,NaN,2017-07-26 00:31:25 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Koda. He is a South Australian decksha...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890006608...,...,1.0,Samoyed,0.957979,True,Pomeranian,0.013884,True,chow,0.008167,True


### 9. *BONUS*
> #### Inconsistent formatting - All columns containing dogs breeds or otherwise (p1, p2, p3) do not have the same case style. Each value found in these columns should either have a capitalized first letter or be all lowercase.

In [22]:
# QA - Issue #5 - Each value found in these columns should either have a capitalized first letter or be all lowercase.
    #update values in columns p1, p2, p3 to be only lowercase
twitter_working_data['p1'] = twitter_working_data['p1'].str.lower()
twitter_working_data['p2'] = twitter_working_data['p2'].str.lower()
twitter_working_data['p3'] = twitter_working_data['p3'].str.lower()
twitter_working_data

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
1,8.921774e+17,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,...,1.0,chihuahua,0.323581,True,pekinese,0.090647,True,papillon,0.068957,True
2,8.918152e+17,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,...,1.0,chihuahua,0.716012,True,malamute,0.078253,True,kelpie,0.031379,True
3,8.916896e+17,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,...,1.0,paper_towel,0.170278,False,labrador_retriever,0.168086,True,spatula,0.040836,False
4,8.913276e+17,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,...,2.0,basset,0.555712,True,english_springer,0.225770,True,german_short-haired_pointer,0.175219,True
5,8.910880e+17,NaN,NaN,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891087950...,...,1.0,chesapeake_bay_retriever,0.425595,True,irish_terrier,0.116317,True,indian_elephant,0.076902,False
6,8.909719e+17,NaN,NaN,2017-07-28 16:27:12 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Meet Jax. He enjoys ice cream so much he gets ...,NaN,NaN,NaN,"https://gofundme.com/ydvmve-surgery-for-jax,ht...",...,1.0,appenzeller,0.341703,True,border_collie,0.199287,True,ice_lolly,0.193548,False
7,8.907292e+17,NaN,NaN,2017-07-28 00:22:40 +0000,"<a href=""http://twitter.com/download/iphone"" r...",When you watch your owner call another dog a g...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890729181...,...,2.0,pomeranian,0.566142,True,eskimo_dog,0.178406,True,pembroke,0.076507,True
8,8.906092e+17,NaN,NaN,2017-07-27 16:25:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Zoey. She doesn't want to be one of th...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890609185...,...,1.0,irish_terrier,0.487574,True,irish_setter,0.193054,True,chesapeake_bay_retriever,0.118184,True
9,8.902403e+17,NaN,NaN,2017-07-26 15:59:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Cassie. She is a college pup. Studying...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890240255...,...,1.0,pembroke,0.511319,True,cardigan,0.451038,True,chihuahua,0.029248,True
10,8.900066e+17,NaN,NaN,2017-07-26 00:31:25 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Koda. He is a South Australian decksha...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890006608...,...,1.0,samoyed,0.957979,True,pomeranian,0.013884,True,chow,0.008167,True


## QA - Issue #6 / QA - Issue #7/ QA - Issue #8:

### QA - Issue #6
> #### Inconsistent formatting - All columns containing status_id should be a string

### QA - Issue #7
> #### Inconsistent formatting - All columns containing tweet_id should be a string

### QA - Issue #8
> #### Inconsistent formatting - All columns containing user_id should be string

In [23]:
twitter_working_data

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
1,8.921774e+17,NaN,NaN,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,NaN,NaN,NaN,https://twitter.com/dog_rates/status/892177421...,...,1.0,chihuahua,0.323581,True,pekinese,0.090647,True,papillon,0.068957,True
2,8.918152e+17,NaN,NaN,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891815181...,...,1.0,chihuahua,0.716012,True,malamute,0.078253,True,kelpie,0.031379,True
3,8.916896e+17,NaN,NaN,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891689557...,...,1.0,paper_towel,0.170278,False,labrador_retriever,0.168086,True,spatula,0.040836,False
4,8.913276e+17,NaN,NaN,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891327558...,...,2.0,basset,0.555712,True,english_springer,0.225770,True,german_short-haired_pointer,0.175219,True
5,8.910880e+17,NaN,NaN,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/891087950...,...,1.0,chesapeake_bay_retriever,0.425595,True,irish_terrier,0.116317,True,indian_elephant,0.076902,False
6,8.909719e+17,NaN,NaN,2017-07-28 16:27:12 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Meet Jax. He enjoys ice cream so much he gets ...,NaN,NaN,NaN,"https://gofundme.com/ydvmve-surgery-for-jax,ht...",...,1.0,appenzeller,0.341703,True,border_collie,0.199287,True,ice_lolly,0.193548,False
7,8.907292e+17,NaN,NaN,2017-07-28 00:22:40 +0000,"<a href=""http://twitter.com/download/iphone"" r...",When you watch your owner call another dog a g...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890729181...,...,2.0,pomeranian,0.566142,True,eskimo_dog,0.178406,True,pembroke,0.076507,True
8,8.906092e+17,NaN,NaN,2017-07-27 16:25:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Zoey. She doesn't want to be one of th...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890609185...,...,1.0,irish_terrier,0.487574,True,irish_setter,0.193054,True,chesapeake_bay_retriever,0.118184,True
9,8.902403e+17,NaN,NaN,2017-07-26 15:59:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Cassie. She is a college pup. Studying...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890240255...,...,1.0,pembroke,0.511319,True,cardigan,0.451038,True,chihuahua,0.029248,True
10,8.900066e+17,NaN,NaN,2017-07-26 00:31:25 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Koda. He is a South Australian decksha...,NaN,NaN,NaN,https://twitter.com/dog_rates/status/890006608...,...,1.0,samoyed,0.957979,True,pomeranian,0.013884,True,chow,0.008167,True


In [24]:
#QA Issue #6 - All columns containing status_id should be a string 
twitter_working_data['in_reply_to_status_id'] = twitter_working_data['in_reply_to_status_id'].astype(str)
twitter_working_data['retweeted_status_id'] = twitter_working_data['retweeted_status_id'].astype(str)

#QA Issue #7 - All columns containing tweet_id should be a string
twitter_working_data['tweet_id'] = twitter_working_data['tweet_id'].astype(str)

#QA Issue #8 - All columns containing user_id should be string
twitter_working_data['in_reply_to_user_id'] = twitter_working_data['in_reply_to_user_id'].astype(str)
twitter_working_data['retweeted_status_user_id'] = twitter_working_data['retweeted_status_user_id'].astype(str)

#### Test

In [25]:
twitter_working_data.head()

,tweet_id,in_reply_to_status_id,in_reply_to_user_id,timestamp,source,text,retweeted_status_id,retweeted_status_user_id,retweeted_status_timestamp,expanded_urls,...,img_num,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog
1,8.92177421306e+17,nan,nan,2017-08-01 00:17:27 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Tilly. She's just checking pup on you....,nan,nan,NaN,https://twitter.com/dog_rates/status/892177421...,...,1.0,chihuahua,0.323581,True,pekinese,0.090647,True,papillon,0.068957,True
2,8.91815181378e+17,nan,nan,2017-07-31 00:18:03 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Archie. He is a rare Norwegian Pouncin...,nan,nan,NaN,https://twitter.com/dog_rates/status/891815181...,...,1.0,chihuahua,0.716012,True,malamute,0.078253,True,kelpie,0.031379,True
3,8.9168955728e+17,nan,nan,2017-07-30 15:58:51 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Darla. She commenced a snooze mid meal...,nan,nan,NaN,https://twitter.com/dog_rates/status/891689557...,...,1.0,paper_towel,0.170278,False,labrador_retriever,0.168086,True,spatula,0.040836,False
4,8.91327558927e+17,nan,nan,2017-07-29 16:00:24 +0000,"<a href=""http://twitter.com/download/iphone"" r...",This is Franklin. He would like you to stop ca...,nan,nan,NaN,https://twitter.com/dog_rates/status/891327558...,...,2.0,basset,0.555712,True,english_springer,0.225770,True,german_short-haired_pointer,0.175219,True
5,8.91087950876e+17,nan,nan,2017-07-29 00:08:17 +0000,"<a href=""http://twitter.com/download/iphone"" r...",Here we have a majestic great white breaching ...,nan,nan,NaN,https://twitter.com/dog_rates/status/891087950...,...,1.0,chesapeake_bay_retriever,0.425595,True,irish_terrier,0.116317,True,indian_elephant,0.076902,False


### T - Issue #2
   > There are extraneous columns that can be deleted - doggo, floofer, pupper, puppo

In [26]:
# T Issue #2 - There are extraneous columns that can be deleted - doggo, floofer, pupper, puppo
twitter_working_data.columns

Index(['tweet_id', 'in_reply_to_status_id', 'in_reply_to_user_id', 'timestamp',
       'source', 'text', 'retweeted_status_id', 'retweeted_status_user_id',
       'retweeted_status_timestamp', 'expanded_urls', 'rating_numerator',
       'rating_denominator', 'name', 'doggo', 'floofer', 'pupper', 'puppo',
       'jpg_url', 'img_num', 'p1', 'p1_conf', 'p1_dog', 'p2', 'p2_conf',
       'p2_dog', 'p3', 'p3_conf', 'p3_dog'],
      dtype='object')

In [27]:
twitter_working_data.drop(columns=['doggo', 'floofer', 'pupper', 'puppo'],inplace=True)

In [28]:
twitter_working_data.columns

Index(['tweet_id', 'in_reply_to_status_id', 'in_reply_to_user_id', 'timestamp',
       'source', 'text', 'retweeted_status_id', 'retweeted_status_user_id',
       'retweeted_status_timestamp', 'expanded_urls', 'rating_numerator',
       'rating_denominator', 'name', 'jpg_url', 'img_num', 'p1', 'p1_conf',
       'p1_dog', 'p2', 'p2_conf', 'p2_dog', 'p3', 'p3_conf', 'p3_dog'],
      dtype='object')

## Storing Data
Save gathered, assessed, and cleaned master dataset to a CSV file named "twitter_archive_master.csv".

In [29]:
#save data to CSV
twitter_working_data.to_csv('twitter_archive_master.csv')

In [30]:
twitter_working_data.columns

Index(['tweet_id', 'in_reply_to_status_id', 'in_reply_to_user_id', 'timestamp',
       'source', 'text', 'retweeted_status_id', 'retweeted_status_user_id',
       'retweeted_status_timestamp', 'expanded_urls', 'rating_numerator',
       'rating_denominator', 'name', 'jpg_url', 'img_num', 'p1', 'p1_conf',
       'p1_dog', 'p2', 'p2_conf', 'p2_dog', 'p3', 'p3_conf', 'p3_dog'],
      dtype='object')

## Analyzing and Visualizing Data
In this section, analyze and visualize your wrangled data. You must produce at least **three (3) insights and one (1) visualization.**

In [31]:
#get count of each breed 
p1_dogs = twitter_working_data['p1']
p2_dogs = twitter_working_data['p2']
p3_dogs = twitter_working_data['p3']

#show top 5 breeds
#where columns p1_dogs, p2_dogs, p3_dogs == True, count the number of times the associated breed appears
dog_breeds_df = pd.concat([p1_dogs,p2_dogs,p3_dogs]).value_counts()
most_commonly_entered = dog_breeds_df.head().to_frame().reset_index()
most_commonly_entered.rename(columns={'index':'breed',0:'count'},inplace=True)
most_commonly_entered.head()

,breed,count
0,golden_retriever,267
1,labrador_retriever,267
2,chihuahua,179
3,pembroke,139
4,cardigan,112


In [32]:
#create dataframe that only includes dog's names and ratings
twitter_dogs_scores = twitter_working_data[['name','rating_numerator','rating_denominator','jpg_url','p1', 'p1_conf', 'p1_dog', 'p2', 'p2_conf',
       'p2_dog', 'p3', 'p3_conf', 'p3_dog']]
#add a column that divides the numerator by the denominator to get the total average score
twitter_dogs_scores['rating_numerator'] = twitter_dogs_scores['rating_numerator'].astype(float)
twitter_dogs_scores['rating_denominator'] = twitter_dogs_scores['rating_denominator'].astype(float)
twitter_dogs_scores['num/denom'] = twitter_dogs_scores['rating_numerator']/twitter_dogs_scores['rating_denominator']

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-vie

In [33]:
#create a column that lists the dog breeds per dog/row
def dog_breeds(df):
    #create column to contain list of dog breeds per row
    df['dog_breeds'] = ''
    
    #iterate through each row
    for index, row in df.iterrows():
        #if p1_dog in each row is True, add value of p1 to new variable dog_breeds
        if row['p1_dog']: 
            dog_breeds = row['p1']
        #iterate for p2
        if row['p2_dog']:
            dog_breeds += ", " + row['p2']
        #iterate for p3
        if row['p3_dog']:
            dog_breeds += ", " + row['p3']
            '''after variable dog_breeds is done being calculated per row, 
                place value in 'dog_breeds' column of row'''
        df.at[index,'dog_breeds'] = dog_breeds

dog1 = np.where(df['p1_dog'],df['p1'])
    dog2 = np.where(df['p2_dog'],df['p2'])
    dog3 = np.where(df['p3_dog'],df['p3']) 
    if len(dog1) =="":
        df['combined_breeds'] = ""
    else: 
        df['combined_breeds'] = (dog1 + ", "); 
    if dog2 == "": 
        df['combined_breeds'] = ""
    else: 
        df['combined_breeds'].append(dog2 + ", ");
    if dog3 == "": 
        df['combined_breeds'] = ""
    else: 
        df['combined_breeds'].append(dog3);                          
    print(df)

In [34]:
dog_breeds(twitter_dogs_scores)

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  after removing the cwd from sys.path.


In [35]:
#sort the num/denom column descending to show the highest scoring dogs
top_dogs = twitter_dogs_scores.sort_values(by='num/denom',ascending=False).head()
top_dogs.reset_index(drop=True, inplace=True)

top_dogs

,name,rating_numerator,rating_denominator,jpg_url,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog,num/denom,dog_breeds
0,Logan,75.0,10.0,https://pbs.twimg.com/media/CurzvFTXgAA2_AP.jpg,pomeranian,0.467321,True,persian_cat,0.122978,False,chow,0.102654,True,7.500000,"pomeranian, chow"
1,Sam,24.0,7.0,https://pbs.twimg.com/media/C0EyPZbXAAAceSc.jpg,golden_retriever,0.871342,True,tibetan_mastiff,0.036708,True,labrador_retriever,0.025823,True,3.428571,"golden_retriever, tibetan_mastiff, labrador_re..."
2,Sophie,27.0,10.0,https://pbs.twimg.com/media/Cswbc2yWcAAVsCJ.jpg,clumber,0.946718,True,cocker_spaniel,0.015950,True,lhasa,0.006519,True,2.700000,"clumber, cocker_spaniel, lhasa"
3,NaN,26.0,10.0,https://pbs.twimg.com/media/CXGaVxOWAAADjhF.jpg,kuvasz,0.438627,True,samoyed,0.111622,True,great_pyrenees,0.064061,True,2.600000,"kuvasz, samoyed, great_pyrenees"
4,NaN,14.0,10.0,https://pbs.twimg.com/media/C_pGRInUwAAmTY_.jpg,lakeland_terrier,0.275242,True,airedale,0.190569,True,teddy,0.102595,False,1.400000,"lakeland_terrier, airedale"


In [36]:
#sort the num/denom column descending to show the lowest scoring dogs
bottom_dogs = twitter_dogs_scores.sort_values(by='num/denom',ascending=True).head()
bottom_dogs.reset_index(drop=True, inplace=True)

bottom_dogs

,name,rating_numerator,rating_denominator,jpg_url,p1,p1_conf,p1_dog,p2,p2_conf,p2_dog,p3,p3_conf,p3_dog,num/denom,dog_breeds
0,NaN,0.0,10.0,https://pbs.twimg.com/media/C5cOtWVWMAEjO5p.jpg,swing,0.967066,False,american_staffordshire_terrier,0.012731,True,staffordshire_bullterrier,0.007039,True,0.0,"border_collie, collie, cardigan, american_staf..."
1,NaN,4.0,20.0,https://pbs.twimg.com/media/CgiFjIpWgAA4wVp.jpg,great_dane,0.246762,True,greater_swiss_mountain_dog,0.126131,True,weimaraner,0.085297,True,0.2,"great_dane, greater_swiss_mountain_dog, weimar..."
2,Tedrick,2.0,10.0,https://pbs.twimg.com/media/CUTILFiWcAE8Rle.jpg,seat_belt,0.200373,False,miniature_pinscher,0.106003,True,schipperke,0.104733,True,0.2,"malamute, golden_retriever, miniature_pinscher..."
3,Crystal,2.0,10.0,https://pbs.twimg.com/media/CWo_T8gW4AAgJNo.jpg,maltese_dog,0.759945,True,toy_poodle,0.101194,True,shih-tzu,0.056037,True,0.2,"maltese_dog, toy_poodle, shih-tzu"
4,Alexanderson,3.0,10.0,https://pbs.twimg.com/media/Cfe5tLWXEAIaoFO.jpg,chihuahua,0.354488,True,carton,0.159672,False,siberian_husky,0.057498,True,0.3,"chihuahua, siberian_husky"


### Insights to gather:
1. The top 5 most commonly entered breeds were golden retriever, labrador retriever, chihuahua, pembroke, and cardigan

2. Highest scoring breeds

3. Lowest scoring breeds

### Insights Results:

In [37]:
#Top 5 most commonly entered breeds
most_commonly_entered

,breed,count
0,golden_retriever,267
1,labrador_retriever,267
2,chihuahua,179
3,pembroke,139
4,cardigan,112


In [38]:
#Highest Scoring Breeds
for dog in top_dogs['dog_breeds']:
    #the function of this for loop is to return every dog breed listed per dog in the top 5 highest scoring dogs
    dog_row = top_dogs.loc[top_dogs['dog_breeds']==dog]
    dog_row_number = dog_row.index[0] +1
    print(dog_row_number, dog)

1 pomeranian, chow
2 golden_retriever, tibetan_mastiff, labrador_retriever
3 clumber, cocker_spaniel, lhasa
4 kuvasz, samoyed, great_pyrenees
5 lakeland_terrier, airedale


In [39]:
#Lowest Scoring Breeds
for dog in bottom_dogs['dog_breeds']:
    #the function of this for loop is to return every dog breed listed per dog in the bottom 5 lowest scoring dogs
    dog_row = bottom_dogs.loc[bottom_dogs['dog_breeds']==dog]
    dog_row_number = dog_row.index[0] +1
    print(dog_row_number, dog)

1 border_collie, collie, cardigan, american_staffordshire_terrier, staffordshire_bullterrier
2 great_dane, greater_swiss_mountain_dog, weimaraner
3 malamute, golden_retriever, miniature_pinscher, schipperke
4 maltese_dog, toy_poodle, shih-tzu
5 chihuahua, siberian_husky


### Visualization

In [40]:
#import matplotlib library for a graph
import matplotlib.pyplot as plt

#create bar plot for most commonly entered dog breeds
plt.barh(data = most_commonly_entered, y = 'breed', width = 'count')
plt.xticks(rotation = 45) #rotate labels by 45 degrees
plt.xlabel("breed")
plt.ylabel("count")
plt.title("Top 5 most commonly entered breeds")
plt.savefig('CommonlyEnteredBreeds.png') # https://stackoverflow.com/questions/13642528/how-to-export-figures-to-files-from-ipython-notebook